In [1]:
from pyecharts.charts import Pie
from pyecharts import options as opts
import pandas as pd
from pyecharts.globals import ThemeType
from IPython.display import HTML, display

# 区域划分映射
REGION_MAPPING = {
    '华北': ['北京市', '天津市', '河北省', '山西省', '内蒙古自治区'],
    '华东': ['上海市', '江苏省', '浙江省', '安徽省', '福建省', '江西省', '山东省'],
    '华南': ['广东省', '广西壮族自治区', '海南省'],
    '西南': ['重庆市', '四川省', '贵州省', '云南省', '西藏自治区'],
    '西北': ['陕西省', '甘肃省', '青海省', '宁夏回族自治区', '新疆维吾尔自治区'],
}

# 指定颜色方案
COLORS = [
    "#e74e69",  # 粉红
    "#f5c353",  # 金黄
    "#11a7ad",  # 青蓝
    "#37a2da",  # 蓝
    "#0070c0",  # 深蓝
    "#7030a0"   # 紫
]

try:
    # 严格读取CSV文件
    df_orders = pd.read_csv('../../数据生成/erp_order.csv', encoding='utf-8-sig')
    
    # 检查必要的列是否存在
    required_columns = ['省份']
    for col in required_columns:
        if col not in df_orders.columns:
            raise ValueError(f"订单数据中缺少必要列: {col}")
    
    # 数据清洗
    if df_orders['省份'].isnull().all():
        raise ValueError("省份数据全部为空")
    
    # 清洗无效省份数据
    df_orders = df_orders[df_orders['省份'].notnull()]
    if df_orders.empty:
        raise ValueError("没有有效的省份数据")
    
    # 映射区域
    def map_region(province):
        for region, provinces in REGION_MAPPING.items():
            if province in provinces:
                return region
        return '其他'
    
    df_orders['区域'] = df_orders['省份'].apply(map_region)
    
    # 计算各区域订单量
    region_orders = df_orders.groupby('区域').size().reset_index(name='订单量')
    
    # 仅保留六大区域
    main_regions = ['华北', '华东', '华南', '西南', '西北']
    region_orders = region_orders[region_orders['区域'].isin(main_regions)]
    
    # 按订单量降序排列
    region_orders = region_orders.sort_values(by='订单量', ascending=False)

    if region_orders.empty:
        raise ValueError("没有符合条件的区域数据")

    # 使用 pyecharts 创建南丁格尔玫瑰图
    pie_chart = (
        Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK, bg_color='#1A1E43', width='1080px', height='600px'))
        .add(
            "",
            [list(z) for z in zip(region_orders['区域'], region_orders['订单量'])],
            radius=["0%", "75%"],
            rosetype="radius",
            label_opts=opts.LabelOpts(
                position="outside",
                color="white",
                formatter="{b}: {d}%"  
            ),
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="全国各区域订单比例分布",
                subtitle=f"所分析的数据量共为{len(df_orders)}",
                title_textstyle_opts=opts.TextStyleOpts(color="white", font_size=30),
                subtitle_textstyle_opts=opts.TextStyleOpts(color="#B0B0B0", font_size=16)
            ),
            legend_opts=opts.LegendOpts(
                orient="vertical", pos_top="15%", pos_right="5%", textstyle_opts=opts.TextStyleOpts(color="white")
            ),
        )
        .set_colors(COLORS)
    )
    
    # 显示图表
    html_content = pie_chart.render_embed()
    display(HTML(html_content))

    # 保存生成的HTML文件
    pie_chart.render("全国各区域订单比例分布.html")
    
except Exception as e:
    print(f"图表生成失败: {str(e)}")
    raise

# 分析

- **华东区域**（红色）：
  占比最高，达到27.0%，表明华东区域在订单分布中占据主导地位，可能与该区域的经济活跃度和人口密度有关。

- **西北区域**（金黄色）：
  占比23.2%，位居第二，显示出西北区域在订单分布中的重要性，可能反映了该区域市场的增长潜力。

- **西南区域**（青蓝色）：
  占比19.4%，位于第三位，表明西南区域的市场表现稳定，可能与该区域的特定消费需求相关。

- **华北区域**（浅蓝色）：
  占比18.9%，略低于西南区域，显示出华北区域的市场活跃度，可能与该区域的经济发展水平有关。

- **华南区域**（深蓝色）：
  占比最小，为11.5%，可能反映了华南区域在订单分布中的相对劣势，需要进一步分析原因。


# 结论

从全国各区域订单比例分布来看，华东和西北区域在市场中表现最为突出，显示出其强大的市场吸引力和用户基础。西南和华北区域也显示出较强的市场潜力，而华南区域的订单比例相对较低，可能需要进一步分析其市场策略和用户定位。